Lab9: OSM and Interactive Web Maps
---
**Xiaofan Liang**  
**Created in 2024; Updated in 2026**


## Learning Objectives
By the end of this practical lab you will be able to:

* Retrieve and process OpenStreetMap (OSM) data with `OSMnx`
* Work with OSM features as `GeoPandas` GeoDataFrames
* Visualize OSM data as a multi-layer interactive map with `Folium`
* Export a Folium map as a static HTML file you can open in any browser
* Host the map on GitHub Pages so anyone can view it with a link

**CHEATSHEETS!** Useful references include the [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html), [GeoPandas documentation](https://geopandas.org/en/stable/docs/user_guide.html), [OSMnx documentation](https://osmnx.readthedocs.io/), [Folium User Guide](https://python-visualization.github.io/folium/latest/user_guide.html), [OSM Wiki Map Features](https://wiki.openstreetmap.org/wiki/Map_features), [Taginfo](https://taginfo.openstreetmap.org/), and [Leaflet Providers Preview](https://leaflet-extras.github.io/leaflet-providers/preview/).

| Function | Task |
|---|---|
| `ox.geocode_to_gdf()` | Retrieve a place boundary by place name and return it as a GeoDataFrame |
| `.total_bounds` | Retrieve a GeoDataFrame bounding box as `(xmin, ymin, xmax, ymax)` |
| `ox.features_from_bbox()` | Retrieve OSM features within a bounding box based on key-value tags |
| `geom_type` | Inspect whether features are points, lines, or polygons |
| `make_valid()` | Repair invalid geometries before mapping or spatial analysis |
| `GeoDataFrame.explore()` | Build an interactive Folium map directly from a GeoDataFrame |
| `explore(m=...)` | Add another GeoDataFrame as a layer onto an existing map |
| `explore(column=..., categorical=True)` | Color features by a category, with an automatic legend |
| `tooltip=` / `popup=` | Show feature info on hover / on click |
| `folium.LayerControl()` | Add controls for turning map layers on and off |
| `fit_bounds()` | Frame the map around a data extent |
| `m.save()` | Export the interactive map as an HTML file |

`OSMnx` queries OpenStreetMap features directly from the Overpass API using a place name or bounding box, and returns them as GeoDataFrames. `Folium` builds interactive Leaflet web maps from those GeoDataFrames and exports them as standalone HTML, which makes them easy to host on a static site like GitHub Pages.


## Environment, Libraries, and Data

Lab 1 created the course virtual environment, and Lab 2 reviewed how to activate it and select its notebook kernel. **Activate the same environment before opening this notebook.**

Install the Lab 9 packages (once) in that environment:
```bash
python3 -m pip install osmnx folium
```

What new packages are for:

* **osmnx** -- queries OpenStreetMap features (buildings, roads, amenities, ...) directly from the Overpass API and returns them as GeoDataFrames.
* **folium** -- builds interactive Leaflet web maps in Python and exports them as standalone HTML.

This lab queries OpenStreetMap data live for the University of Michigan North Campus area -- no local data files are needed.


In [16]:
%pip install osmnx folium

Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install mapclassify

Note: you may need to restart the kernel to use updated packages.


In [18]:
%pip install osmnx folium

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from shapely import make_valid

# Cache OSMnx requests so repeated runs don't keep re-downloading the same data.
ox.settings.use_cache = True

In [6]:
ox.settings.overpass_url = "https://overpass.private.coffee/api"
ox.settings.requests_timeout = 180

print("Current server:", ox.settings.overpass_url)

Current server: https://overpass.private.coffee/api


## Retrieve OSM Data on UMich North Campus

OSM data has three geometry types: **Node**, **Way**, and **Relation**. A node can be a point of interest, but it can also be a vertex that forms the geometry of a polygon. A way can be a line or polygon, such as a road or building. A relation is a collection of nodes, ways, and other relations, such as a bus line or administrative boundary. In this lab, we mostly work with nodes and ways.

There are two steps for downloading OSM data:

1. Determine the **bounding box** for the download.
2. Tell OSM what **features** (key-value tags) you want -- amenities, roads, buildings -- and convert the result into a GeoDataFrame.


### Step 1: Determine the Bounding Box

An OSM query starts with a bounding box. You can pass a place name as a string: `ox.geocode_to_gdf()` uses a geocoding service to find the boundary associated with a place name and returns it as a GeoDataFrame, and `.total_bounds` then gives you `(xmin, ymin, xmax, ymax)`. Always double-check the returned boundary since many places share similar names.

You can also supply four coordinates directly as `(xmin, ymin, xmax, ymax)`. Click the bottom-left corner of your area of interest in Google Maps (or another map interface) for `xmin`/`ymin`, and the top-right corner for `xmax`/`ymax`. Since "North Campus" isn't a well-defined place name in OSM, we use a numeric bounding box for it instead.


In [19]:
# Geocode Ann Arbor and the University of Michigan for comparison -- neither perfectly
# matches North Campus, which is why we use a numeric bounding box below instead.
aa = ox.geocode_to_gdf('Ann Arbor, Michigan, USA')
print('Ann Arbor total_bounds:', aa.total_bounds)

um = ox.geocode_to_gdf('University of Michigan, Ann Arbor, Michigan, USA')
print('UMich total_bounds:', um.total_bounds)

# Roughly the bottom-left and top-right corners of UM North Campus.
# OSMnx expects bbox values in the order (left, bottom, right, top).
bbox = (-83.725706, 42.287305, -83.704518, 42.302508)
bbox

Ann Arbor total_bounds: [-83.799572   42.222668  -83.675807   42.3238941]
UMich total_bounds: [-83.7502236  42.242226  -83.6649743  42.3125148]


(-83.725706, 42.287305, -83.704518, 42.302508)

In [20]:
import folium

center_lat = (bbox[1] + bbox[3]) / 2
center_lon = (bbox[0] + bbox[2]) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=15,
    tiles="OpenStreetMap"
)

m

### Step 2: Add OSM Features and Convert to GeoDataFrames

OSM features are stored as **key-value** (or **feature-value**) pairs. To retrieve restaurants, you'd use `key='amenity'`, `value='restaurant'` -- written in OSMnx as a dictionary, `{"amenity": "restaurant"}`. Passing `True` instead of a value (e.g. `{"building": True}`) retrieves **all** values under that key; passing a list (e.g. `{"amenity": ["restaurant", "cafe"]}`) retrieves several values in one query.

**How do you find out what key-value pairs exist?**

- The [OSM Wiki Map Features page](https://wiki.openstreetmap.org/wiki/Map_features) covers most commonly used pairs, with definitions for each.
- [Taginfo](https://taginfo.openstreetmap.org/) shows how many instances of a key-value pair exist -- useful for checking whether a tag is common enough, or is only used in a specific local context.
- Google your interest together with "OSM." Searching "restaurant in OSM" turns up the [amenity=restaurant](https://wiki.openstreetmap.org/wiki/Tag:amenity%3Drestaurant) page directly.

We retrieve three layers this way: **buildings**, **roads**, and **amenities**.


#### Buildings

`{"building": True}` retrieves every building-tagged feature. The result can mix point and polygon geometries (a node may carry building-related tags even though the polygon is what we want), so we keep only polygons. OSM polygons occasionally have minor topology errors (such as not closing correctly); `make_valid()` from `shapely` repairs those before mapping. We also drop down to just the `geometry` column. OSM features come back with dozens of tag columns we don't need for this map, and keeping them increases the size of the exported file.


In [29]:

# LAB 9: Download OSM features and build the interactive map

import osmnx as ox
import geopandas as gpd
import folium
import pandas as pd

# Set a more reliable Overpass server and increase timeout
ox.settings.overpass_url = "https://overpass-api.de/api"
ox.settings.timeout = 180
ox.settings.use_cache = True

# Use the existing bounding box, but make it smaller
west, south, east, north = bbox

center_lon = (west + east) / 2
center_lat = (south + north) / 2

factor = 0.60

small_bbox = (
    center_lon - (east - west) * factor / 2,
    center_lat - (north - south) * factor / 2,
    center_lon + (east - west) * factor / 2,
    center_lat + (north - south) * factor / 2
)

# Retrieve buildings, roads, and amenities in ONE request
tags = {
    "building": True,
    "highway": True,
    "amenity": True
}

osm = ox.features_from_bbox(small_bbox, tags=tags)

print("Total OSM features:", len(osm))

# Clean geometries
osm = osm.reset_index()
osm = osm[osm.geometry.notna()].copy()
osm["geometry"] = osm.geometry.make_valid()

# Separate the three layers
building = osm[osm["building"].notna()].copy()
roads = osm[osm["highway"].notna()].copy()
amenities = osm[osm["amenity"].notna()].copy()

# Keep only appropriate geometry types
building = building[
    building.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

roads = roads[
    roads.geometry.geom_type.isin(["LineString", "MultiLineString"])
].copy()

# Convert amenities to points
amenity_point = amenities.copy()
amenity_point["geometry"] = amenity_point.geometry.representative_point()

# Make sure tooltip columns exist
for layer in [building, roads, amenity_point]:
    for col in ["name", "building", "highway", "amenity"]:
        if col not in layer.columns:
            layer[col] = None

print("Buildings:", len(building))
print("Roads:", len(roads))
print("Amenities:", len(amenity_point))

# Create interactive map
m = folium.Map

Total OSM features: 1594
Buildings: 168
Roads: 980
Amenities: 300


In [28]:
import osmnx as ox

# Increase the time allowed for downloading data
ox.settings.requests_timeout = 300

# Turn on caching
ox.settings.use_cache = True

print("OSM settings updated!")

OSM settings updated!


#### Roads

Roads are stored under the `highway` key. In OSM, `highway` covers all road-like features -- residential streets, service roads, footways, cycleways, paths -- not just highways in the everyday sense. `{"highway": True}` retrieves all of them; we keep the line geometries and, again, trim to just the columns the map actually uses.


In [21]:
# retrieve all features within the bounding box that are tagged as highways. 
roads = ox.features_from_bbox(bbox, tags={'highway': True})

# print the count of each geometry type 
print(roads.geom_type.value_counts())

# only keep line geometries
roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])]

# Inspect the most common highway values in this area before trimming columns.
print(roads['highway'].value_counts().head(8))

# reset row index and keep the highway and geometry columns while dropping other columns; we only want to display limited information in the interactive map. 
roads = roads[['highway', 'geometry']].reset_index(drop=True)
roads.head()

LineString    1946
Point          342
Polygon         44
Name: count, dtype: int64
highway
footway         1271
service          358
steps             93
cycleway          63
unclassified      35
tertiary          33
path              32
residential       30
Name: count, dtype: int64


,highway,geometry
0,unclassified,"LINESTRING (-83.7127 42.30039, -83.71282 42.30..."
1,tertiary,"LINESTRING (-83.71297 42.29036, -83.71313 42.2..."
2,service,"LINESTRING (-83.72461 42.29701, -83.72525 42.2..."
3,service,"LINESTRING (-83.71025 42.29122, -83.71035 42.2..."
4,service,"LINESTRING (-83.7127 42.29956, -83.71278 42.29..."


#### Amenities

We retrieve several food-and-drink amenity types in one query by passing a list of values under `amenity`, then keep point geometries. Many OSM points are missing a `name`, which would leave the map's tooltip/popup blank. We drop those here to keep the demo map focused on labeled points, though in a real analysis you'd usually keep unnamed points too. 


In [22]:
# Define OSM tags to search for specific amenity types (food/drink, etc.)
tags = {
    'amenity': [
        'restaurant', 'bar', 'biergarten', 'fast_food',
        'food_court', 'cafe', 'pub', 'ice_cream',
    ]
}

# Retrieve all features within the bounding box that match the tags.
amenity = ox.features_from_bbox(bbox, tags=tags)

# Print the count of each geometry type
print(amenity.geom_type.value_counts())

# Convert non-point geometries (Polygon, MultiPolygon) to centroids before filtering columns.
amenity_points = amenity[amenity.geom_type == "Point"].copy()
amenity_polys = amenity[amenity.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
amenity_polys["geometry"] = amenity_polys.geometry.centroid
amenity_points = pd.concat([amenity_points, amenity_polys], ignore_index=True)  # ignore_index=True avoids duplicate or misaligned indices when combining

# Drop entries missing the 'name' column (so only named places remain)
amenity_points = amenity_points.dropna(subset=['name'])

# Keep only the relevant columns: name, amenity type, geometry; reset the index
amenity_point = amenity_points[['name', 'amenity', 'geometry']].reset_index(drop=True)

# Display a count for each amenity type present in the results
print(amenity_point['amenity'].value_counts())

# If you see a UserWarning about geographic CRS, ignore it. We only use centroids for mapping, which are unaffected by the CRS.

Point      17
Polygon     7
Name: count, dtype: int64
amenity
restaurant    11
fast_food      8
cafe           5
Name: count, dtype: int64


/var/folders/yn/99yn_ph92gg13ckljnvsnxx80000gn/T/ipykernel_12509/2658919970.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  amenity_polys["geometry"] = amenity_polys.geometry.centroid


## Visualize OSM Data

**Folium** is aimed at *interactive* web maps: popups, pan/zoom, a shareable HTML file. **GeoPandas + Matplotlib** (used throughout earlier labs) is the usual choice for *static*, publication-quality figures. `GeoDataFrame.explore()` is a fast bridge between the two: it builds a Folium map directly from a GeoDataFrame, in one line, for a quick interactive look before you invest in custom styling.


In [23]:
# Fastest possible interactive preview: one GeoDataFrame, one line.
amenity_point.explore(color='blue', tiles='OpenStreetMap')

### Get a Free CARTO API Key

CARTO's basemap tiles (used below for their light, low-contrast look) now require a free API key -- unauthenticated requests come back with a visible "API KEY REQUIRED" watermark across the map. Getting a key is quick and does **not** require creating a CARTO account:

1. Go to [carto.com/basemaps/apikey](https://carto.com/basemaps/apikey).
2. Fill out the short form: your email, the domain where you'll host the map (your GitHub Pages domain is fine, e.g. `your-username.github.io`), and a one-line project description (e.g. "class assignment").
3. CARTO emails you a key, usually within a few minutes. The free tier covers 5 million tile requests/month -- far more than this lab needs.
4. Paste your key into `CARTO_API_KEY` below.

**Why CARTO and not Mapbox?** Mapbox also issues free API tokens, and token creation there is instant (no email wait at all) -- but Mapbox requires a credit card on file to unlock its free tier, even if you're never charged. CARTO's signup needs no card. For a whole class getting a key, that's worth more than saving a few minutes.


In [10]:
# Replace with your own key from carto.com/basemaps/apikey 
CARTO_API_KEY = 'cb1_4kla_2_044855deaedd03ce0de2aa56'

### Build the full map

`GeoDataFrame.explore()` builds a Folium map directly from a GeoDataFrame. Pass `m=` to add another GeoDataFrame onto a map you already built, instead of hand-building `folium.Map()` plus `folium.GeoJson()`/`CircleMarker()` calls yourself. We chain three layers onto one map:

- **buildings** -- solid dark grey polygons (this call creates the map)
- **roads** -- black lines
- **amenities** -- circle markers colored automatically by the `amenity` column (`column='amenity', categorical=True`), with an automatic legend, and a tooltip/popup showing the name and type

Each `.explore()` call's `name=` becomes one entry in the layer control. 

Folium can do much more than this! Explore its [User Guide](https://python-visualization.github.io/folium/latest/user_guide.html) for many more powerful examples, such as:
- [Choropleth maps](https://python-visualization.github.io/folium/latest/user_guide/geojson/choropleth.html): color polygons or regions by a data variable (e.g. population density, rates, etc.)
- [Heatmaps](https://python-visualization.github.io/folium/latest/user_guide/plugins/heatmap.html): visualize point intensity or density with smooth color gradients
- [TimestampedGeoJson](https://python-visualization.github.io/folium/latest/user_guide/plugins/timestamped_geojson.html): show how patterns change over time with a time slider
- Custom [popups](https://python-visualization.github.io/folium/latest/user_guide/ui_elements/popups.html), [icons](https://python-visualization.github.io/folium/latest/user_guide/ui_elements/icons.html), and [image overlays](https://python-visualization.github.io/folium/latest/user_guide/raster_layers/image_overlay.html)
- [GeoJSON with style functions and highlight features](https://python-visualization.github.io/folium/latest/user_guide/geojson/geojson.html)

You'll find many quick examples and patterns in the documentation!


In [30]:
# CartoDB Positron: a light, low-contrast basemap that keeps colored overlays readable.
# The {z}/{x}/{y}/{r} placeholders are filled in by Leaflet itself at render time,
# not by Python, so they stay as plain text here -- only the API key is substituted.

# required attribution for using the map tiles
tile_url = "https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png"
attr = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

# Build the basemap as its own TileLayer
base_tiles = folium.TileLayer(tiles=tile_url, attr=attr, name='CartoDB Positron', control=False)

# Buildings first: this call creates the map itself (m), styled as solid dark grey polygons with no border.
m = building.explore(
    tiles=base_tiles,
    color='darkgrey',
    style_kwds={'fillOpacity': 0.8, 'weight': 0},
    name='buildings',
)

# Roads: pass m= to add this layer onto the SAME map instead of creating a new one.
roads.explore(
    m=m,
    color='black',
    style_kwds={'weight': 1.5, 'opacity': 0.7},
    name='roads',
)

# Amenities: color automatically by the 'amenity' column (categorical=True gives each
# unique value its own color from the cmap), with an automatic legend, and a
# tooltip (hover) / popup (click) showing the name and type for each point.
amenity_point.explore(
    m=m,
    column='amenity',
    categorical=True,
    cmap=['red', 'blue', 'darkgreen'],
    marker_kwds={'radius': 5},
    tooltip=['name', 'amenity'],
    popup=['name', 'amenity'],
    legend=True,
    name='amenities',
)

# Let users toggle the buildings, roads, and amenities layers independently.
folium.LayerControl(collapsed=False).add_to(m)

m

In [32]:
print("building exists:", "building" in globals())
print("buildings exists:", "buildings" in globals())

building exists: True
buildings exists: False


Zoom, pan, and click markers to open popups. Use the layer control (top right) to toggle buildings, roads, and amenities.


## Export and Host on GitHub Pages

`m.save()` writes a standalone HTML file. If you double-click on the file exported in your local folder, you can see and interact with it on your browser. This is a local copy, meaning that you can see it, but you cannot share a link for others to see. To make it sharable, you will have to host it online and we recommend **GitHub Pages** because it provides free hosting service. 


In [33]:
html_file = 'lab9_map.html'
m.save(html_file)
print('Saved:', html_file)

Saved: lab9_map.html


### Step 1: Create a GitHub repository

1. Log in to [GitHub](https://github.com) and click **New repository**.
2. Choose a name (e.g. `lab9-osm-map`), set visibility to **Public**, and create the repo.

### Step 2: Add your HTML file and enable GitHub Pages

**Option A -- upload in the browser**

1. In the repo, under the **Quick setup** ssection click the **upload an existing file** link and drag `lab9_map.html` into the upload box that opens.
2. Commit.
3. **Settings** -> **Pages** -> **Deploy from a branch** -> branch **main**, folder **/ (root)** -> Save.
4. After a short wait, open `https://<your-username>.github.io/<repo-name>/lab9_map.html`.

**Option B -- Git from your computer**

```bash
cd "PATH TO YOUR CURRENT LAB FOLDER"
git clone https://github.com/your-username/lab9-osm-map.git
cd lab9-osm-map
#  manually copy lab9_map.html into this folder, then type the following commands
git add lab9_map.html
git commit -m "Add Folium map"
git push origin main
```

Then enable Pages as in Option A.

### Step 3: Share the link

Use `https://<your-username>.github.io/<repo-name>/lab9_map.html` in assignments. **Tip:** rename the file to `index.html` in the repo root if you want the map at `https://<your-username>.github.io/<repo-name>/` with no filename.

## Practice Questions

### Q1: Retrieve a new OSM feature

Choose another OSM feature on North Campus -- bicycle parking, bus stops, parking lots, benches, or another amenity -- and retrieve it with `ox.features_from_bbox()`, following the same two-step process as above (the `bbox` from Step 1 still applies). Use the OSM Wiki or Taginfo to find the right key-value pair.

### Q2: Add it to the map

Add the new feature to the map above as another chained layer (besides amenities, roads, and building). Give it a color distinct from the amenities layer, and a tooltip or popup showing at least one useful attribute. Make sure it shows up as a new legend item and keep all other map aesthetics (e.g., legend style, basemap style, etc.) the same. 

### Q3: Export and host

Export your updated map as an HTML file, host it on GitHub Pages (or another static host), and submit the public URL for the quiz.

## Additional Resources

[OSMnx documentation](https://osmnx.readthedocs.io/), [Pyrosm documentation](https://pyrosm.readthedocs.io/) (an alternative for file-based OSM extracts), [GeoPandas documentation](https://geopandas.org/), [Folium User Guide](https://python-visualization.github.io/folium/latest/user_guide.html), [OSM Wiki Map Features](https://wiki.openstreetmap.org/wiki/Map_features), [Taginfo](https://taginfo.openstreetmap.org/), and [Leaflet Providers Preview](https://leaflet-extras.github.io/leaflet-providers/preview/).


In [9]:
print("OSMnx version:", ox.__version__)
print("Overpass URL:", ox.settings.overpass_url)
print("Overpass endpoint:", getattr(ox.settings, "overpass_endpoint", "not available"))

OSMnx version: 2.1.1
Overpass URL: https://overpass.private.coffee/api
Overpass endpoint: not available


In [16]:
print("bbox exists:", "bbox" in globals())
print("m exists:", "m" in globals())
print("ox exists:", "ox" in globals())
print("building exists:", "building" in globals())

bbox exists: False
m exists: False
ox exists: True
building exists: False


In [24]:
# Switch to an alternate OpenStreetMap server

# Retrieve bicycle parking locations
bicycle_parking = ox.features_from_bbox(
    bbox,
    tags={"amenity": "bicycle_parking"}
)

print("Bicycle parking locations found:", len(bicycle_parking))

# Add bicycle parking as a purple layer
if not bicycle_parking.empty:
    bicycle_parking.explore(
        m=m,
        color="purple",
        marker_kwds={"radius": 6},
        tooltip=["amenity"],
        popup=["amenity"],
        name="bicycle parking",
    )

# Add layer controls
folium.LayerControl(collapsed=False).add_to(m)

m

ReadTimeout: HTTPSConnectionPool(host='overpass.private.coffee', port=443): Read timed out. (read timeout=180)

In [15]:
print("building:", "building" in globals())

building: False
